# 📧 Gmail Reader & Email Parser Demo

This notebook demonstrates how to connect to Gmail using **Google OAuth 2.0**, fetch emails, parse their MIME payloads, and extract key fields:

- 👤 **Sender** (Display Name & Email Address)
- 🌐 **Domain of the Sender**
- 📌 **Subject**
- 🕒 **Time of the Email** (Date Header)
- 📎 **Has Attachment** (Boolean flag, filenames, sizes, MIME types)
- 📝 **Email Content** (Decoded text content)
- 📬 **Recipient** (`To` header)
- 🔄 **Reply-To** (and sender vs reply-to mismatch)
- 🔗 **Embedded URLs** (Links extracted from plain text & HTML)
- 🛡️ **Security & Authentication Headers** (SPF, DKIM, DMARC)

In [1]:
%load_ext autoreload
%autoreload 2

## 1. Environment & Imports

Resolve project path and import the Gmail reader and parser modules from `scamlens`.

In [2]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Ensure project root and src/ are in sys.path
project_root = Path("..").resolve() if Path("../src").exists() else Path(".").resolve()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Load .env
env_file = project_root / ".env"
if env_file.exists():
    load_dotenv(env_file)

from scamlens.core.config import get_settings
from scamlens.extractors.email import (
    EmailMessage,
    GmailAuthManager,
    GmailReader,
    parse_gmail_message,
)

settings = get_settings()
creds_file = project_root / settings.gmail_credentials_file
token_file = project_root / settings.gmail_token_file

print(f"Project root:     {project_root}")
print(f"Credentials path: {creds_file} (exists: {creds_file.exists()})")
print(f"Token path:       {token_file} (exists: {token_file.exists()})")

Project root:     /home/vishnu/ScamLens-AI
Credentials path: /home/vishnu/ScamLens-AI/credentials.json (exists: True)
Token path:       /home/vishnu/ScamLens-AI/token.json (exists: True)


## 2. Connect to Gmail via Google Auth

Initialize `GmailReader` using `GmailAuthManager`.
- If `token.json` is present, it authenticates automatically without prompting.
- If not yet authenticated, it initiates the OAuth consent flow and saves `token.json` for future sessions.

In [3]:
auth_manager = GmailAuthManager(
    credentials_file=str(creds_file),
    token_file=str(token_file),
)

reader = GmailReader(auth_manager=auth_manager)

# Initialize service connection
service = reader.service
print("✅ Successfully connected to Gmail API!")

✅ Successfully connected to Gmail API!


## 3. Query & Fetch Emails from Inbox

You can customize the search query to fetch specific emails:
- `is:unread` - Unread emails
- `newer_than:7d` - Received within the last 7 days
- `label:INBOX` - In the Inbox
- `has:attachment` - Emails with file attachments
- `""` (empty string) - Most recent emails

In [4]:
# ==============================================================================
# Query Configuration:
# ------------------------------------------------------------------------------
# How to choose what emails to read:
#
# - Latest UNREAD emails (any category):    QUERY = "is:unread"
# - LATEST emails in Inbox (read + unread): QUERY = "label:INBOX"
# - Primary category only (skip promo/social): QUERY = "category:primary"
# - Updates tab (orders, statements):       QUERY = "category:updates"
# - Promotions tab:                         QUERY = "category:promotions"
# - Social tab (Reddit, LinkedIn):          QUERY = "category:social"
# - Specific folder/label (e.g. SPAM):      QUERY = "label:SPAM"
# - With file attachments only:             QUERY = "has:attachment"
# ==============================================================================

QUERY = "is:unread"   # Try "label:INBOX" for latest emails, or "category:primary"
MAX_EMAILS = 3        # Number of emails to fetch

print(f"Searching emails with query: '{QUERY}' (limit: {MAX_EMAILS})...")
messages = reader.fetch_messages(query=QUERY, max_results=MAX_EMAILS)
print(f"Fetched {len(messages)} email(s).\n")


Searching emails with query: 'is:unread' (limit: 3)...
Fetched 3 email(s).



## 4. Print & Inspect Extracted Fields for Each Email

Here we print all the key parsed fields for each email:
- **Sender** (`sender_name` & `sender_email`)
- **Domain of the Sender** (`sender_domain`)
- **Subject** (`subject`)
- **Time of the Email** (`date`)
- **Recipient** (`recipient`)
- **Reply-To** (`reply_to`)
- **Has Attachment** (`has_attachment` and attachment list)
- **Email Content** (`body_cleaned` / `body_plain`)
- **Embedded Links / URLs** (`extracted_urls`)
- **Security Auth Headers** (`spf_status`, `dkim_status`, `dmarc_status`)

In [5]:
for idx, msg in enumerate(messages, start=1):
    print("=" * 80)
    print(f"EMAIL #{idx} OF {len(messages)}")
    print("=" * 80)
    print(f"Sender:               {msg.sender}")
    print(f"Display Name:         {msg.sender_name or '(None)'}")
    print(f"Email Address:        {msg.sender_email or '(None)'}")
    print(f"Domain of the Sender: {msg.sender_domain or '(None)'}")
    print(f"Subject:              {msg.subject}")
    print(f"Time of the Email:    {msg.date}")
    print(f"Recipient:            {msg.recipient}")
    if msg.reply_to:
        print(f"Reply-To:             {msg.reply_to}")
    print(f"Has Attachment:       {'YES' if msg.has_attachment else 'NO'}")
    if msg.has_attachment:
        for a_idx, att in enumerate(msg.attachments, start=1):
            size_kb = att.size_bytes / 1024 if att.size_bytes else 0
            print(f"  - Attachment #{a_idx}: {att.filename} ({size_kb:.1f} KB, {att.mime_type})")
    print(f"Authentication:       SPF={msg.spf_status.upper()} | DKIM={msg.dkim_status.upper()} | DMARC={msg.dmarc_status.upper()}")
    print(f"Embedded Links count: {len(msg.extracted_urls)}")
    print("-" * 80)
    print("Email Content:")
    content = msg.body_cleaned or msg.body_plain or msg.snippet
    print(content)
    print("=" * 80 + "\n")


EMAIL #1 OF 3
Sender:               Reddit <noreply@redditmail.com>
Display Name:         Reddit
Email Address:        noreply@redditmail.com
Domain of the Sender: redditmail.com
Subject:              "Selling urgently"
Time of the Email:    Wed, 18 Nov 2026 15:33:53 +0000
Recipient:            vishnuday1475@gmail.com
Has Attachment:       NO
Authentication:       SPF=PASS | DKIM=PASS | DMARC=PASS
Embedded Links count: 59
--------------------------------------------------------------------------------
Email Content:
Reddit r/HyderabadUsedItems: Asus Vivobook 16x

Ryzen 5-5600H

16 GB DDR4 3200 RAM

512 GB SSD

Fingerprint scanner

Backlit keyboard

Location: Madhapur, Hyderabad

Almost 2year old laptop

Bill not available all original... ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ 

## 5. Summary Table

View all fetched emails in an organized table summary.

In [6]:
# Print a clean aligned summary table
header_fmt = "{:<4} {:<30} {:<24} {:<35} {:<12} {:<10}"
row_fmt    = "{:<4} {:<30} {:<24} {:<35} {:<12} {:<10}"

print(header_fmt.format("#", "Sender Email", "Sender Domain", "Subject", "Attachment?", "Links"))
print("-" * 120)

for idx, m in enumerate(messages, start=1):
    subj = (m.subject[:32] + "..") if len(m.subject) > 34 else m.subject
    sender = (m.sender_email[:27] + "..") if len(m.sender_email) > 29 else m.sender_email
    domain = (m.sender_domain[:22] + "..") if len(m.sender_domain) > 24 else m.sender_domain
    has_att = f"YES ({len(m.attachments)})" if m.has_attachment else "NO"
    
    print(row_fmt.format(idx, sender, domain, subj, has_att, len(m.extracted_urls)))

#    Sender Email                   Sender Domain            Subject                             Attachment?  Links     
------------------------------------------------------------------------------------------------------------------------
1    noreply@redditmail.com         redditmail.com           "Selling urgently"                  NO           59        
2    updates@updates.indmoney.com   updates.indmoney.com     Anthropic goes public tomorrow! ..  NO           11        
3    info@connect.netmeds.com       connect.netmeds.com      Made with ❤️ for you                NO           17        


## 6. Inspect a Specific Email by Message ID

Use this helper to inspect any specific email and view its complete un-truncated content and raw metadata dictionary.

In [7]:
def inspect_email_by_id(message_id: str) -> None:
    """Fetch a single message by ID and display its full details and content."""
    msg = reader.get_message(message_id)
    
    print(f"ID:             {msg.id}")
    print(f"Thread ID:      {msg.thread_id}")
    print(f"Sender:         {msg.sender}")
    print(f"Sender Domain:  {msg.sender_domain}")
    print(f"Subject:        {msg.subject}")
    print(f"Date:           {msg.date}")
    print(f"Has Attachment: {msg.has_attachment}")
    print("\n--- Full Content ---")
    print(msg.body_cleaned or msg.body_plain or msg.snippet)

# If we have messages, inspect the first one:
if messages:
    first_id = messages[0].id
    print(f"Inspecting Message ID: {first_id}\n")
    inspect_email_by_id(first_id)
else:
    print("No messages available to inspect.")

Inspecting Message ID: 1a1ef4b059f08ce8

ID:             1a1ef4b059f08ce8
Thread ID:      1a1ef4b059f08ce8
Sender:         Reddit <noreply@redditmail.com>
Sender Domain:  redditmail.com
Subject:        "Selling urgently"
Date:           Wed, 18 Nov 2026 15:33:53 +0000
Has Attachment: False

--- Full Content ---
Reddit r/HyderabadUsedItems: Asus Vivobook 16x

Ryzen 5-5600H

16 GB DDR4 3200 RAM

512 GB SSD

Fingerprint scanner

Backlit keyboard

Location: Madhapur, Hyderabad

Almost 2year old laptop

Bill not available all original... ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ VarietyOk6501 r/HyderabadUsedItems u/Kanna_GS • 3h ago Selling ur